## Unit 08. 插值、微分與積分之運算 — 課堂作業
- 課程編號: CHEXXXX
- 學年度: 114下
- 上課時間: 每週四 09:00-12:00
-
- 指導教授: ＯＯＯ 教授
- 學生姓名: ＯＯＯ
- 學號: m12345678
- email帳號: fcu.m12345678@gmail.com

---
### 0. 環境設定

In [1]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit08_Homework'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit08'
        OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR      = OUTPUT_DIR / 'figs'
    else:
        print("⚠️ 找不到雲端 ChemE-3502 路徑，請確認資料夾名稱是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR      = OUTPUT_DIR / 'figs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

✓ 偵測到 Local 環境

✓ Notebook工作目錄: d:\MyGit\ChemE-3502\Unit08
✓ 結果輸出目錄: d:\MyGit\ChemE-3502\Unit08\outputs\Unit08_Homework
✓ 圖檔輸出目錄: d:\MyGit\ChemE-3502\Unit08\outputs\Unit08_Homework\figs


---
### 1. 載入套件

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from scipy.interpolate import interp1d, CubicSpline
from scipy.integrate import quad, trapezoid, simpson

plt.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'axes.unicode_minus': False,
})

print("✓ 套件載入完成")
print(f"  numpy      版本: {np.__version__}")
import scipy
print(f"  scipy      版本: {scipy.__version__}")
import matplotlib
print(f"  matplotlib 版本: {matplotlib.__version__}")

✓ 套件載入完成
  numpy      版本: 1.23.5
  scipy      版本: 1.15.2
  matplotlib 版本: 3.10.8


---
### **I. 練習題: 一維插值 — 丙酮蒸氣之熱傳導係數**

**問題描述 (習題3-5-1)**

丙酮蒸氣在不同溫度下之熱傳導係數量測數據如下 (Hanna and Sandall, 1995)：

| 溫度 T (°F) | 32 | 115 | 212 | 363 |
|:----------:|:---:|:---:|:---:|:---:|
| k (Btu/hr·ft·°F) | 0.0057 | 0.0074 | 0.0099 | 0.0147 |

試利用不同的一維插值方法分析此數據。

In [3]:
# ========================================
# 已知數據 (勿修改此區塊)
# ========================================
# 溫度數據 (°F)
T_data = np.array([32.0, 115.0, 212.0, 363.0])

# 熱傳導係數數據 (Btu/hr·ft·°F)
k_data = np.array([0.0057, 0.0074, 0.0099, 0.0147])

print("丙酮蒸氣熱傳導係數原始數據:")
print(f"{'溫度 T (°F)':<20} {'k (Btu/hr·ft·°F)':<20}")
print("-" * 42)
for T, k in zip(T_data, k_data):
    print(f"{T:<20.1f} {k:<20.4f}")

print("\n欲求插值之溫度點:")
T_query = np.array([300.0])
print(f"  T = {T_query[0]}°F  (求 k 值)")
print("\n逆向插值目標:")
print(f"  k = 0.008 Btu/hr·ft·°F  (求對應溫度)")

丙酮蒸氣熱傳導係數原始數據:
溫度 T (°F)            k (Btu/hr·ft·°F)    
------------------------------------------
32.0                 0.0057              
115.0                0.0074              
212.0                0.0099              
363.0                0.0147              

欲求插值之溫度點:
  T = 300.0°F  (求 k 值)

逆向插值目標:
  k = 0.008 Btu/hr·ft·°F  (求對應溫度)


### **作業任務：**

**1. 插值方法比較**
- 使用 `scipy.interpolate.interp1d()` 以 `linear`、`quadratic`、`cubic` 三種方法建立插值函數
- 使用 `scipy.interpolate.CubicSpline()` 建立三次樣條插值函數
- 在 $T = 32{\sim}363$ °F 範圍內繪製各方法之插值曲線，並標示原始數據點

**2. 求解 $T = 300$ °F 之熱傳導係數**
- 利用上述四種插值方法，分別估計 $T = 300$ °F 時的 $k$ 值
- 將結果整理成比較表格
- 參考答案：約 $0.0127$ Btu/hr·ft·°F

**3. 逆向插值 — 求 $k = 0.008$ 時之溫度**
- 利用其中一種插值方法，配合 `scipy.optimize.brentq()` 或直接對插值函數求逆，求得 $k = 0.008$ 時的對應溫度
- 參考答案：約 $138.28$ °F

**4. 結果輸出**
- 將圖檔儲存至 `FIG_DIR / 'Fig1_Acetone_k_Interpolation.png'`

In [ ]:
# ===== 請在此處撰寫你的程式碼 =====


---
### **II. 練習題: 數值微分 — 動力學數據分析**

**問題描述 (習題3-5-11)**

在等溫下進行 $A \rightarrow B$ 之反應，測得反應物濃度數據如下 (Levenspiel, 1999)：

| 時間 (秒) | 0 | 20 | 40 | 60 | 120 | 180 | 300 |
|:--------:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| $C_A$ (mol/L) | 10 | 8 | 6 | 5 | 3 | 2 | 1 |

假設反應速率式為：

$$
-r_A = k C_A^n
$$

試根據實驗數據，利用**微分法**確認速率式中之反應階數 $n$ 與速率常數 $k$ 的值。

**微分法原理：**

對速率式兩邊取對數：

$$
\ln(-r_A) = \ln(k) + n \ln(C_A)
$$

以 $\ln(-r_A)$ 對 $\ln(C_A)$ 作圖，斜率即為 $n$，截距即為 $\ln(k)$。

In [4]:
# ========================================
# 已知數據 (勿修改此區塊)
# ========================================
# 時間數據 (秒)
t_data = np.array([0.0, 20.0, 40.0, 60.0, 120.0, 180.0, 300.0])

# 反應物濃度 CA (mol/L)
CA_data = np.array([10.0, 8.0, 6.0, 5.0, 3.0, 2.0, 1.0])

print("A → B 批次反應實驗數據:")
print(f"{'時間 t (s)':<15} {'CA (mol/L)':<15}")
print("-" * 32)
for t, CA in zip(t_data, CA_data):
    print(f"{t:<15.1f} {CA:<15.1f}")

A → B 批次反應實驗數據:
時間 t (s)        CA (mol/L)     
--------------------------------
0.0             10.0           
20.0            8.0            
40.0            6.0            
60.0            5.0            
120.0           3.0            
180.0           2.0            
300.0           1.0            


### **作業任務：**

**1. 計算數值微分 — 反應速率**
- 使用 `numpy.gradient(CA_data, t_data)` 計算 $dC_A/dt$（非均勻間距版本）
- 取反應速率 $-r_A = -dC_A/dt$（注意：速率取正值）
- 注意邊界點（起始與終點）之處理

**2. 繪製 $\ln(-r_A)$ vs. $\ln(C_A)$ 圖**
- 以 $\ln(C_A)$ 為橫軸，$\ln(-r_A)$ 為縱軸作圖
- 使用 `numpy.polyfit()` 進行線性回歸，求得斜率 $n$ 與截距 $\ln(k)$
- 在圖上標示回歸直線

**3. 確認反應動力學參數**
- 輸出粗估之反應階數 $n$ 與速率常數 $k$
- 將 $n$ 取最近整數後，重新計算修正的 $k$ 值（使用最小平方法）
- 參考答案：粗估 $n \approx 1.285$，$k \approx 5.936 \times 10^{-3}$；取 $n = 1$ 時，$k \approx 1.075 \times 10^{-2}$

**4. 驗證圖**
- 以實驗數據的 $C_A - t$ 曲線與利用所求參數重建的模型預測曲線作比較圖
- 將圖檔儲存至 `FIG_DIR / 'Fig2_Kinetics_Analysis.png'`

In [ ]:
# ===== 請在此處撰寫你的程式碼 =====


---
### **III. 練習題: 數值積分 — 追蹤劑響應與滯留時間分布**

**問題描述 (習題3-5-4)**

在研究反應器混合特性時，將追蹤劑於某瞬間注入反應器，記錄出口濃度 $C(t)$。下表為某實驗所得之量測值 (Constantinides and Mostoufi, 1999)：

| 時間 (秒) | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|:--------:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| C(t) (mg/L) | 0 | 2 | 4 | 7 | 6 | 5 | 2 | 1 | 0 |

**相關公式：**

滯留時間分布函數 (RTD)：

$$
E(t) = \frac{C(t)}{\int_{0}^{\infty} C(t)\, dt}
$$

累積分布函數 (CDF)：

$$
F(t) = \int_{0}^{t} E(t)\, dt
$$

平均滯留時間：

$$
t_m = \int_{0}^{\infty} t \cdot E(t)\, dt
$$

RTD 函數之變異數：

$$
\sigma^2 = \int_{0}^{\infty} (t - t_m)^2 \cdot E(t)\, dt
$$

In [5]:
# ========================================
# 已知數據 (勿修改此區塊)
# ========================================
# 時間數據 (秒) - 等間距
t_rtd = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])

# 追蹤劑出口濃度 C(t) (mg/L)
C_rtd = np.array([0.0, 2.0, 4.0, 7.0, 6.0, 5.0, 2.0, 1.0, 0.0])

print("追蹤劑出口濃度量測數據:")
print(f"{'時間 t (s)':<15} {'C(t) (mg/L)':<15}")
print("-" * 32)
for t, C in zip(t_rtd, C_rtd):
    print(f"{t:<15.1f} {C:<15.1f}")

print(f"\n數據點數: {len(t_rtd)}")
print(f"時間範圍: {t_rtd[0]} ~ {t_rtd[-1]} 秒")

追蹤劑出口濃度量測數據:
時間 t (s)        C(t) (mg/L)    
--------------------------------
0.0             0.0            
1.0             2.0            
2.0             4.0            
3.0             7.0            
4.0             6.0            
5.0             5.0            
6.0             2.0            
7.0             1.0            
8.0             0.0            

數據點數: 9
時間範圍: 0.0 ~ 8.0 秒


### **作業任務：**

**1. 計算 RTD 函數 $E(t)$**
- 使用 `scipy.integrate.trapezoid(C_rtd, t_rtd)` 計算 $\int_0^\infty C(t)\,dt$
- 計算 $E(t) = C(t) / \int_0^\infty C(t)\,dt$
- 驗證：確認 $\int_0^\infty E(t)\,dt = 1.0$

**2. 計算累積分布函數 $F(t)$**
- 利用梯形積分，逐點計算累積積分 $F(t_i) = \int_0^{t_i} E(t)\,dt$（可用迴圈或 `numpy.cumsum` 輔助）
- 確認 $F(t_{max}) = 1.0$

**3. 計算平均滯留時間 $t_m$**
- 使用 `scipy.integrate.trapezoid(t_rtd * E, t_rtd)` 計算 $t_m = \int_0^\infty t \cdot E(t)\,dt$
- 參考答案：$t_m = 3.667$ 秒

**4. 計算 RTD 變異數 $\sigma^2$**
- 計算 $\sigma^2 = \int_0^\infty (t - t_m)^2 \cdot E(t)\,dt$
- 參考答案：$\sigma^2 = 2.222$

**5. 繪製結果圖（雙圖）**
- 左圖：$E(t)$ 隨時間之分布（填充面積），標示 $t_m$
- 右圖：$F(t)$ 累積分布曲線
- 在圖中標示計算結果 $t_m$ 和 $\sigma^2$
- 將圖檔儲存至 `FIG_DIR / 'Fig3_RTD_Analysis.png'`

In [ ]:
# ===== 請在此處撰寫你的程式碼 =====


---
## IV. 額外加分作業 (自由繳交)
- 學生可自由選擇是否繳交加分作業。(下週上課前完成 email 電子檔)
- 分數會加在最後總成績上，每份作業加 0.1 ~ 1.0 分。
- 加分作業不一定要全部都做完，想繳交至少完成其中一個問題即可。
- 請務必自行完成！你可以問 AI，問同學，問學長姊，但一定要自行【吸收】【執行】【整理】結果！
- 如果系統自動比對發現作業內容（與他人雷同，原創性低，抄襲比例過高），作業的分數會 0 分。

---
### **加分題: 等溫管式反應器之設計 (習題3-5-9)**

在一個等溫管式反應器中，進行如下之氣相反應 (黃，2004)：

$$
2A \rightarrow S, \quad -r_A = 116\left(P_A^2 - \frac{1}{1.27} P_S\right)
$$

**已知條件：**

| 參數 | 數值 |
|:---|:---:|
| 反應溫度 $T$ | 638 °C |
| 壓力 $P$ | 1 atm |
| 反應器內徑 $d$ | 10 cm |
| 惰性氣體/A 莫耳比 $\alpha$ | 0.5 |
| 進料速率 $F_0$ | 10 kg-mol/h |

**管式反應器設計方程式（積分形式）：**

$$
L = \frac{F_{A0}}{A_c} \int_0^{x_A} \frac{dx_A}{-r_A}
$$

其中截面積 $A_c = \frac{1}{4}\pi d^2$，各組分分壓為：

$$
P_A = P \cdot \frac{1-x_A}{1 - \frac{1}{2}x_A + \alpha}, \quad P_S = P \cdot \frac{\frac{1}{2}x_A}{1 - \frac{1}{2}x_A + \alpha}
$$

**作業要求：**

1. 計算轉化率 $x_A = 0.5$ 時，所需的反應器管長 $L$
   - 參考答案：$L = 42.52 \text{ m}$

2. 若管長固定為 $L = 5 \text{ m}$，計算出口轉化率 $x_{Af}$
   - 提示：計算不同 $x_{Af}$ 對應的管長，再以插值求解
   - 參考答案：$x_{Af} = 0.229$

3. 繪製轉化率 $x_A$ 與管長 $L$ 的關係圖（$x_A$ 從 0 到 0.8）
4. 將圖檔儲存至 `FIG_DIR / 'Fig4_PFR_Design.png'`

In [ ]:
# ===== 請在此處撰寫你的程式碼 =====


---
## 💭 思考題

1. 在插值問題中，數據點越多是否代表插值結果越精確？高次多項式插值可能產生什麼問題（Runge 現象）？

2. `linear`、`cubic` 和 `CubicSpline` 三種插值方法有何本質差異？在外插（超出數據範圍）時，各方法的可靠性如何？

3. 在利用 `numpy.gradient()` 計算數值微分時，數據的等間距與非等間距有何不同？邊界點的處理方式為何？

4. 使用梯形積分法 (`trapezoid`) 與 `scipy.integrate.quad()` 的主要差異是什麼？分別適用於哪類問題？

5. 在 RTD 分析中，$\sigma^2$ 的大小反映了什麼物理意義？$\sigma^2$ 趨近於 0 代表什麼流動特性？

6. 在等溫管式反應器設計中，若進料中含有惰性氣體，對積分被積分函數有什麼影響？為什麼工業上有時需要加入惰性氣體？

---
# 想對老師說的話